# Restaurant RAG — step by step

Run **one cell at a time**. Each step depends on the cells above it.

Need `OPENAI_API_KEY` in `.env`.

## 1. Imports and API key

In [ ]:
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from docx import Document as WordDocument
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

PROJECT_ROOT = Path(r"C:\dev\ai-workplace\restaurant-ai-assistant")
PROMPT_DIR = PROJECT_ROOT / "prompts"
MENU_PATH = PROJECT_ROOT / "data" / "menu.docx"

load_dotenv(PROJECT_ROOT / ".env")

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Project root:", PROJECT_ROOT)
print("Menu exists:", MENU_PATH.exists())
print("Prompts exist:", (PROMPT_DIR / "system.txt").exists(), (PROMPT_DIR / "summarize.txt").exists(), (PROMPT_DIR / "rewrite.txt").exists())

## 2. Load the Word menu

`word_doc` is python-docx. It is **not** a LangChain document yet.

In [ ]:
word_doc = WordDocument(MENU_PATH)
lines = [p.text.strip() for p in word_doc.paragraphs if p.text.strip()]

print("File:", MENU_PATH)
print("Paragraphs:", len(lines))
print("First 8 lines:")
for line in lines[:8]:
    print(" -", line)

## 3. Chunk: one LangChain Document per menu item

In [ ]:
CATEGORIES = [
    "Simply Pizza Range",
    "Classic Pizza Range",
    "Supreme Pizza Range",
    "Premium Pizza Range",
    "Chef's Special Pizza Range",
    "Vito's Special - XL Size Pizza",
    "Extra Toppings for Pizza",
    "Calzone",
    "Salad",
    "Pasta",
    "Lasanga",
    "Dessert",
    "Drinks",
]
SKIP_PREFIXES = ("Size:", "Choice of", "Folded", "Vito's Restaurant Menu")

docs = []
category = None
range_price = None
notes = []
i = 0

while i < len(lines):
    line = lines[i]
    if line in CATEGORIES:
        category, range_price, notes = line, None, []
        i += 1
        continue
    if line.startswith("Range Price:"):
        range_price = line.replace("Range Price:", "").strip()
        i += 1
        continue
    if line.startswith(SKIP_PREFIXES):
        notes.append(line)
        i += 1
        continue

    name = line
    price, ingredients = range_price, None
    i += 1
    while i < len(lines) and lines[i] not in CATEGORIES:
        if lines[i].startswith("Price:"):
            price = lines[i].replace("Price:", "").strip()
            i += 1
        elif lines[i].startswith("Ingredients/Description:"):
            ingredients = lines[i].replace("Ingredients/Description:", "").strip()
            i += 1
        else:
            break

    parts = [f"Category: {category}", f"Item: {name}"]
    if price:
        parts.append(f"Price: {price}")
    if ingredients:
        parts.append(f"Ingredients: {ingredients}")
    if notes:
        parts.append("Notes: " + " ".join(notes))

    docs.append(
        Document(
            page_content="\n".join(parts),
            metadata={"category": category, "item": name},
        )
    )

print(len(docs), "chunks")

## 4. Inspect chunks

Expect about 82 items, 17 drinks, 4 desserts. If a heading became an item, fix `CATEGORIES` / `SKIP_PREFIXES` and rerun step 3.

In [ ]:
print(Counter(d.metadata["category"] for d in docs))
print("\n--- first pizza ---")
print(docs[0].page_content)
print("\n--- first drink ---")
print(next(d.page_content for d in docs if d.metadata["category"] == "Drinks"))

## 5. OpenAI embeddings

This calls OpenAI (`text-embedding-3-small`). It is not the chat LLM.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
sample_vector = embeddings.embed_query("mojito")
print("Embedding length:", len(sample_vector))

## 6. Vector store

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)
print("Indexed", len(docs), "chunks")

## 7. Retriever only (no LLM yet)

Confirm drinks do not pull whole dessert/topping sections.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
retrieved = retriever.invoke("what drinks do you have")

for i, doc in enumerate(retrieved, start=1):
    print(f"{i}. [{doc.metadata['category']}] {doc.metadata['item']}")

## 8. LLM — ChatOpenAI

This is the model that writes the answer. Test it **without** the menu first.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=120,
)

raw = llm.invoke("Say hello in one short sentence.")
print(raw.content)

## 9. Prompt template

Loaded from `prompts/system.txt` and `prompts/human.txt`.
`{context}` = retrieved menu items. `{question}` = user question.

In [ ]:
system_text = (PROMPT_DIR / "system.txt").read_text(encoding="utf-8")
human_text = (PROMPT_DIR / "human.txt").read_text(encoding="utf-8")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_text),
        ("human", human_text),
    ]
)

print(system_text)
print("---")
print(prompt)

## 10. Turn retrieved docs into one context string

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

context_preview = format_docs(retrieved)
print(context_preview[:800])

## 11. Build the chain

```
question → retriever → format_docs → prompt → llm → string
```

In [ ]:
chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(type(chain))

## 12. Run the chain

In [ ]:
print(chain.invoke("What drinks do you have?"))

In [ ]:
print(chain.invoke("Do you have a vegetarian pizza?"))

In [ ]:
print(chain.invoke("Hi"))

## 13. NEW cells — LangGraph memory (do not replace steps 1–12)

The chain above stays as it is. These cells add a **second** path.

**Agreement:** do not keep every message forever.

- Keep the **last 4 messages** in full (so “that / it” still works)
- **Summarize the rest** into one running `summary`
- Same `thread_id` = same conversation

`pip install langgraph` if needed. Reuse `retriever`, `llm`, `format_docs`, and `system_text` from above.

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

print("langgraph imports ok")

## 14. Graph state

- `messages` — recent turns (appended each invoke)
- `summary` — compressed older chat
- `search_query` — standalone search text (“that” resolved)
- `context` — retrieved menu chunks

In [ ]:
MESSAGE_THRESHOLD = 6
KEEP_LAST = 4


class GraphState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str
    search_query: str
    context: str

## 15. Nodes

`summarize` → `rewrite` → `retrieve` → `generate`

When `len(messages) > 6`:
- **summarize** messages except the last 4
- **delete** those old messages from the thread
- **keep** the last 4 raw + the summary string

In [ ]:
summarize_text = (PROMPT_DIR / "summarize.txt").read_text(encoding="utf-8").strip()
rewrite_text = (PROMPT_DIR / "rewrite.txt").read_text(encoding="utf-8").strip()


def maybe_summarize(state: GraphState) -> dict:
    messages = state["messages"]
    if len(messages) <= MESSAGE_THRESHOLD:
        return {}

    old, recent = messages[:-KEEP_LAST], messages[-KEEP_LAST:]
    existing = state.get("summary") or ""
    transcript = "\n".join(f"{m.type}: {m.content}" for m in old)
    result = llm.invoke(
        [
            SystemMessage(content=summarize_text),
            HumanMessage(content=f"Previous summary:\n{existing}\n\nOlder messages:\n{transcript}"),
        ]
    )
    print(f"Summarized {len(old)} older messages; keeping last {len(recent)} raw.")
    return {
        "summary": result.content,
        "messages": [RemoveMessage(id=m.id) for m in old],
    }


def rewrite_query(state: GraphState) -> dict:
    messages = state["messages"]
    last = messages[-1].content
    if len(messages) == 1 and not state.get("summary"):
        return {"search_query": last}

    recent = "\n".join(f"{m.type}: {m.content}" for m in messages)
    payload = (
        f"Latest user message:\n{last}\n\n"
        f"Recent messages (most recent last). Resolve it/that from these first:\n{recent}\n\n"
        f"Older summary (background only; do not use for it/that unless the latest message clearly refers back):\n"
        f"{state.get('summary') or '(none)'}"
    )
    result = llm.invoke(
        [
            SystemMessage(content=rewrite_text),
            HumanMessage(content=payload),
        ]
    )
    print("search_query:", result.content.strip())
    return {"search_query": result.content.strip()}


def retrieve_menu(state: GraphState) -> dict:
    docs = retriever.invoke(state["search_query"])
    return {"context": format_docs(docs)}


def generate_answer(state: GraphState) -> dict:
    system = system_text
    if state.get("summary"):
        system += f"\n\nEarlier conversation summary:\n{state['summary']}"
    system += f"\n\nMenu context:\n{state['context']}"
    result = llm.invoke([SystemMessage(content=system), *state["messages"]])
    return {"messages": [AIMessage(content=result.content)]}


print("nodes load prompts from files; keep last", KEEP_LAST)


## 16. Compile the graph

`MemorySaver` is enough in the notebook. Same `thread_id` = same conversation.

In [ ]:
builder = StateGraph(GraphState)
builder.add_node("summarize", maybe_summarize)
builder.add_node("rewrite", rewrite_query)
builder.add_node("retrieve", retrieve_menu)
builder.add_node("generate", generate_answer)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "rewrite")
builder.add_edge("rewrite", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

graph = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "demo"}}


def chat(question: str, thread_id: str = "demo") -> str:
    cfg = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=cfg,
    )
    answer = result["messages"][-1].content
    print(f"[{thread_id}] Q: {question}")
    print(f"[{thread_id}] A: {answer}\n")
    return answer


print(type(graph))

## 17. Same thread — follow-up should remember “that”

Run after steps 1–10 and 13–16. The second question should refer to Beef BBQ pizza.

In [ ]:
chat("How much is Beef BBQ pizza?", "demo")
chat("Is that spicy?", "demo")

## 18. Different thread — should not know about Beef BBQ

In [ ]:
chat("Is that spicy?", "other-guest")

## 19. Trip the summary threshold

Each turn adds 2 messages (human + AI). After a few more turns, `maybe_summarize` should fold older ones into `summary`. Inspect state after this cell.

In [ ]:
chat("Do you have a vegetarian pizza?", "demo")
chat("What drinks do you have?", "demo")
chat("Hi", "demo")

state = graph.get_state({"configurable": {"thread_id": "demo"}})
print("summary:\n", state.values.get("summary"))
print("\nraw messages still stored:", len(state.values["messages"]))
for m in state.values["messages"]:
    print("-", m.type, ":", str(m.content)[:80])